In [ ]:
import pandas as pd
import numpy as np
from transformers import BertModel, BertTokenizer
import torch
from torch import nn
from itertools import combinations
from sklearn.feature_extraction.text import TfidfVectorizer
from skmultilearn.problem_transform import BinaryRelevance
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, multilabel_confusion_matrix,accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

In [5]:
import pandas as pd
data_os = pd.read_csv('../OurData/os.csv')
data_qt = pd.read_csv('../OurData/qt.csv')
data_android = pd.read_csv('../OurData/android.csv')

In [6]:
from sklearn.metrics import multilabel_confusion_matrix, matthews_corrcoef
import numpy as np
def calculate_mcc(y_true, y_pred):
    """
    Calculate the macro and micro average Matthews Correlation Coefficient (MCC)
    for multilabel classification.

    Parameters:
    y_true (np.ndarray): True labels, shape (n_samples, n_labels)
    y_pred (np.ndarray): Predicted labels, shape (n_samples, n_labels)

    Returns:
    macro_avg_mcc (float): Macro average MCC
    micro_avg_mcc (float): Micro average MCC
    """
    # Compute multilabel confusion matrix
    mcm = multilabel_confusion_matrix(y_true, y_pred)

    # Calculate MCC for each label
    mccs = []
    TP = TN = FP = FN = 0
    for cm in mcm:
        tn, fp, fn, tp = cm.ravel()
        mcc = matthews_corrcoef(
            [1]*tp + [0]*tn + [1]*fn + [0]*fp,
            [1]*tp + [1]*fp + [0]*fn + [0]*tn
        )
        mccs.append(mcc)

        # Aggregate TP, TN, FP, FN for micro MCC calculation
        TP += tp
        TN += tn
        FP += fp
        FN += fn

    # Macro average MCC
    macro_avg_mcc = np.mean(mccs)

    # Compute MCC from aggregated TP, TN, FP, FN
    numerator = (TP * TN) - (FP * FN)
    denominator = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    micro_avg_mcc = numerator / denominator if denominator != 0 else 0

    return macro_avg_mcc, micro_avg_mcc

In [8]:
train_data = pd.concat([data_android, data_qt], ignore_index=True)
test_data = data_os

In [9]:
Label = ["bug","test","feature","resource","deprecat","merge","refactor"]

In [10]:
def get_feature_combinations(features):
    all_combinations = []
    # Define mandatory features that should always be present
    mandatory_features = {'Title', 'Description'}
    # Extract optional features by removing the mandatory ones
    optional_features = [feature for feature in features if feature not in mandatory_features]

    # Start from 0 to generate combinations including none of the optional features
    for i in range(0, len(optional_features) + 1):
        for combo in combinations(optional_features, i):
            # Add mandatory features to each combination
            all_combinations.append(mandatory_features.union(combo))

    return all_combinations

In [11]:
features = ['Description', 'Title', 'OwnerName', 'ChangedFiles']
# Get all valid combinations of features, ensuring 'Title' and 'Description' are always included
feature_combinations = get_feature_combinations(features)

In [12]:
param_grids = {
    'DecisionTree': {
        'max_features': ['auto', 'sqrt', 'log2'],
        'ccp_alpha': [0.1, .01, .001],
        'max_depth': [5, 6, 7, 8, 9],
        'criterion': ['gini', 'entropy']
    },
    'LogisticRegression': {
        'C': [0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'saga']
    },
    'RandomForest': {
        'n_estimators': [100, 200],
        'max_depth': [5, 10, None],
        'criterion': ['gini', 'entropy']
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'probability': [True]
    },
    'GradientBoosting': {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5]
    },
    'XGBoost': {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5],
        'colsample_bytree': [0.3, 0.7],
        'subsample': [0.8, 1.0]
    }
}

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')  

def encode_texts(texts):
    inputs = tokenizer(
        texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,
    )
    with torch.no_grad():
        outputs = bert_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1).numpy()
    return embeddings


models = {
    'DecisionTree': DecisionTreeClassifier(random_state=1024),
    # 'LogisticRegression': LogisticRegression(random_state=1024),
    # 'RandomForest': RandomForestClassifier(random_state=1024),
    # 'SVC': SVC(random_state=1024),
    # 'GradientBoosting': GradientBoostingClassifier(random_state=1024),
    # 'XGBoost': XGBClassifier(random_state=1024, use_label_encoder=False, eval_metric='logloss')
}

results = []

results = []

# Iterate over all models and perform grid search for each
for model_name, model in models.items():
    print(f"Processing model: {model_name}")
    
    param_grid = param_grids[model_name]  # Get the parameter grid for the current model
    
    for combo in feature_combinations:
        print(f"Processing combination: {combo}")
        
        # Prepare the combined data column
        train_data['combined_data'] = train_data[list(combo)].astype(str).agg(' '.join, axis=1)
        test_data['combined_data'] = test_data[list(combo)].astype(str).agg(' '.join, axis=1)

        # Encode the texts using BERT embeddings
        X_train = encode_texts(train_data['combined_data'].tolist())
        X_test = encode_texts(test_data['combined_data'].tolist())

        y_train = train_data[Label]
        y_test = test_data[Label]

        # Perform GridSearchCV
        classifier_gsearch = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, n_jobs=-1, verbose=1)
        classifier_gsearch.fit(X_train, y_train)

        # Best parameters from the grid search
        best_params = classifier_gsearch.best_params_

        # Create the final classifier with the best parameters
        best_model = model.set_params(**best_params)
        classifier = BinaryRelevance(classifier=best_model)
        classifier.fit(X_train, y_train)

        # Make predictions and predict probabilities
        predictions = classifier.predict(X_test)
        y_proba = classifier.predict_proba(X_test)

        # Convert predictions and probabilities to arrays for metrics calculation
        y_test_array = y_test.values
        y_pred_array = predictions.toarray()
        y_proba_array = y_proba.toarray()

        # Calculate micro and macro metrics
        macro_mcc, micro_mcc = calculate_mcc(y_test_array, y_pred_array)
        macro_auc = roc_auc_score(y_test_array, y_proba_array, average='macro')
        micro_auc = roc_auc_score(y_test_array, y_proba_array, average='micro')

        for i, label in enumerate(Label):
            # Metrics for each label
            mcc = matthews_corrcoef(y_test_array[:, i], y_pred_array[:, i])
            auc = roc_auc_score(y_test_array[:, i], y_proba_array[:, i])
            recall = recall_score(y_test_array[:, i], y_pred_array[:, i])
            precision = precision_score(y_test_array[:, i], y_pred_array[:, i])
            f1 = f1_score(y_test_array[:, i], y_pred_array[:, i])
            accuracy = accuracy_score(y_test_array[:, i], y_pred_array[:, i])
            label_count = np.sum(y_test_array[:, i])

            results.append({
                'Model': model_name,
                'Feature Combination': ', '.join(combo),
                'Label': label,
                'MCC': mcc,
                'AUC': auc,
                'Recall': recall,
                'Precision': precision,
                'F1 Score': f1,
                'Accuracy': accuracy,
                'Label Count': label_count,
                'Best Params': best_params
            })

        # Compute macro and micro averages
        macro_recall = recall_score(y_test_array, y_pred_array, average='macro')
        macro_precision = precision_score(y_test_array, y_pred_array, average='macro')
        macro_f1 = f1_score(y_test_array, y_pred_array, average='macro')
        macro_accuracy = accuracy_score(y_test_array, y_pred_array)

        micro_recall = recall_score(y_test_array, y_pred_array, average='micro')
        micro_precision = precision_score(y_test_array, y_pred_array, average='micro')
        micro_f1 = f1_score(y_test_array, y_pred_array, average='micro')
        micro_accuracy = accuracy_score(y_test_array, y_pred_array)

        # Add macro and micro metrics to the results
        results.append({
            'Model': model_name,
            'Feature Combination': ', '.join(combo),
            'Label': 'Macro',
            'MCC': macro_mcc,
            'AUC': macro_auc,
            'Recall': macro_recall,
            'Precision': macro_precision,
            'F1 Score': macro_f1,
            'Accuracy': macro_accuracy,
            'Best Params': best_params
        })
        results.append({
            'Model': model_name,
            'Feature Combination': ', '.join(combo),
            'Label': 'Micro',
            'MCC': micro_mcc,
            'AUC': micro_auc,
            'Recall': micro_recall,
            'Precision': micro_precision,
            'F1 Score': micro_f1,
            'Accuracy': micro_accuracy,
            'Best Params': best_params
        })

# Convert the results into a DataFrame for analysis
metrics_df = pd.DataFrame(results)

# Print the metrics table
print(metrics_df)

# Optionally, save to a CSV file
metrics_df.to_csv('metrics_results.csv', index=False)